# NIH ChestXray14 Data Pipeline

This notebook handles the entire process: environment setup, downloading 50% of the dataset, extracting images, and preprocessing metadata.

In [ ]:
# Step 1: Initialize Environment
import os
import sys
import pandas as pd
import urllib.request
import tarfile
from tqdm.notebook import tqdm
from glob import glob

# Define paths
DATA_DIR = 'data'
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
METADATA_PATH = os.path.join(DATA_DIR, 'Data_Entry_2017_v2020.csv')
CLEANED_METADATA = os.path.join(DATA_DIR, 'metadata_cleaned.csv')

os.makedirs(IMAGES_DIR, exist_ok=True)
print("Environment initialized. Path to images:", os.path.abspath(IMAGES_DIR))

In [ ]:
# Step 2: Download and Extract 50% of Batches

links = [
    'https://nihcc.box.com/shared/static/vfk49d74nhbxq3nqjg0900w5nvkorp5c.gz',
    'https://nihcc.box.com/shared/static/i28rlmbvmfjbl8p2n3ril0pptcmcu9d1.gz',
    'https://nihcc.box.com/shared/static/f1t00wrtdk94satdfb9olcolqx20z2jp.gz',
    'https://nihcc.box.com/shared/static/0aowwzs5lhjrceb3qp67ahp0rd1l1etg.gz',
    'https://nihcc.box.com/shared/static/v5e3goj22zr6h8tzualxfsqlqaygfbsn.gz',
    'https://nihcc.box.com/shared/static/asi7ikud9jwnkrnkj99jnpfkjdes7l6l.gz'
]

def download_and_extract(url, dest_dir, batch_idx):
    filename = f"images_{batch_idx:03d}.tar.gz"
    filepath = os.path.join(dest_dir, filename)
    
    if not os.path.exists(filepath):
        print(f"\nDownloading {filename}...")
        # Simple retrieval with basic visual feedback
        urllib.request.urlretrieve(url, filename=filepath)
    
    print(f"Extracting {filename}...")
    with tarfile.open(filepath, 'r:gz') as tar:
        tar.extractall(path=dest_dir)
    
    os.remove(filepath)
    print(f"Done with batch {batch_idx}.")

pbar = tqdm(links, desc="Processing Batches")
for i, link in enumerate(pbar):
    try:
        download_and_extract(link, DATA_DIR, i+1)
    except Exception as e:
        print(f"Error processing batch {i+1}: {e}")

In [ ]:
# Step 3: Process Metadata (Create sync point)

if os.path.exists(METADATA_PATH):
    df = pd.read_csv(METADATA_PATH)
    df_cleaned = df[['Image Index', 'Finding Labels']]
    df_cleaned.to_csv(CLEANED_METADATA, index=False)
    
    # Check what we have locally
    if os.path.exists(IMAGES_DIR):
        avail_list = os.listdir(IMAGES_DIR)
        available_images = {x: os.path.join(IMAGES_DIR, x) for x in avail_list if x.endswith('.png')}
        df_cleaned['path'] = df_cleaned['Image Index'].map(available_images.get)
        df_final = df_cleaned.dropna(subset=['path']).reset_index(drop=True)
        
        print(f"Total metadata rows: {len(df_cleaned)}")
        print(f"Images found locally and mapped: {len(df_final)}")
    else:
        print("Images directory not found yet.")
else:
    print("Original metadata CSV not found at", METADATA_PATH)

In [ ]:
# Step 4: Verification
if 'df_final' in locals() and not df_final.empty:
    print("--- Data Preview ---")
    display(df_final.head())
else:
    print("Running Step 2 & 3 is required to see progress here.")